# Clustering Track — Gene Expression Cancer RNA-Seq (UCI)
**Goal:** group tumour samples by gene-expression profile with **no labels used during fitting**.
The known cancer types are only used *afterwards* to validate the clusters (rubric §3.3).

Algorithms: **K-Means** (elbow curve) and **Agglomerative Hierarchical** (dendrogram, linkage comparison).
Metrics: Silhouette, Davies-Bouldin, Calinski-Harabasz. Visuals: PCA 2-D (mandatory) and t-SNE.



In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.manifold import TSNE
from sklearn.metrics import (silhouette_score, davies_bouldin_score, calinski_harabasz_score,
                             adjusted_rand_score, normalized_mutual_info_score)
from scipy.cluster.hierarchy import linkage, dendrogram

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid", palette="colorblind")
plt.rcParams["figure.dpi"] = 100


Load and Audit

In [ ]:
# The UCI zip (TCGA-PANCAN-HiSeq-801x20531) contains data.csv (samples x genes) and labels.csv (Class).
# Put the unzipped folder anywhere under ../data/ — the loader searches for the two files.
DATA_DIR = Path("../data")
data_path = next(DATA_DIR.rglob("data.csv"), None)
labels_path = next(DATA_DIR.rglob("labels.csv"), None)
if data_path is None:
    raise FileNotFoundError("data.csv not found under ../data/ — unzip the UCI gene-expression download there.")

expr = pd.read_csv(data_path, index_col=0)
labels = pd.read_csv(labels_path, index_col=0)["Class"].loc[expr.index] if labels_path else None

print("Expression matrix (samples x genes):", expr.shape)
print("Missing values:", int(expr.isnull().sum().sum()))
print("Duplicate samples:", int(expr.duplicated().sum()))
print("dtypes:", expr.dtypes.value_counts().to_dict())
if labels is not None:
    print("\nKnown classes (held back — used for validation only):")
    print(labels.value_counts())


**Observation:** *(samples vs genes — far more features than samples; note missing values, class balance, and why this "p >> n" shape makes scaling + PCA essential.)*

## EDA — gene-level variance

In [ ]:
gene_var = expr.var()
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(np.log10(gene_var[gene_var > 0]), bins=60)
axes[0].set_xlabel("log10(variance of gene)"); axes[0].set_ylabel("Number of genes")
axes[0].set_title("Distribution of gene variance")

if labels is not None:
    labels.value_counts().plot(kind="bar", ax=axes[1])
    axes[1].set_title("Known tumour types (for later validation)")
    axes[1].set_xlabel("Class"); axes[1].set_ylabel("Samples")
plt.tight_layout(); plt.savefig("../models/clu_eda.png", bbox_inches="tight"); plt.show()

print("Zero-variance genes:", int((gene_var == 0).sum()))


**Observation:** *(most genes have low variance — only a subset varies enough to separate tumour types, which motivates variance filtering.)*

## Preprocessing — variance filter, scaling, PCA(50)


In [ ]:
# Drop constant genes, keep the most variable ones, then standardise.
# (Clustering has no train/test split, so scaling on the full matrix is not leakage.)
X = expr.loc[:, gene_var > 0]
TOP_GENES = min(2000, X.shape[1])
top_genes = X.var().sort_values(ascending=False).head(TOP_GENES).index
X = X[top_genes]
print("Genes kept:", X.shape[1])

X_scaled = StandardScaler().fit_transform(X)

# PCA to 50 components: removes noise, makes distance-based clustering meaningful in high dimensions
pca50 = PCA(n_components=min(50, X_scaled.shape[1]), random_state=RANDOM_STATE)
X_pca = pca50.fit_transform(X_scaled)

plt.figure(figsize=(7, 4))
plt.plot(np.cumsum(pca50.explained_variance_ratio_), marker="o")
plt.xlabel("Number of components"); plt.ylabel("Cumulative explained variance")
plt.title("PCA — variance retained by the 50 components used for clustering")
plt.tight_layout(); plt.savefig("../models/clu_pca_variance.png", bbox_inches="tight"); plt.show()
print(f"Variance retained by {pca50.n_components_} components: {pca50.explained_variance_ratio_.sum():.3f}")


## K-Means — Elbow curve (and silhouette) to choose k

In [ ]:
k_range = range(2, 11)
inertias, sils = [], []
for k in k_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(X_pca)
    inertias.append(km.inertia_)
    sils.append(silhouette_score(X_pca, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(list(k_range), inertias, marker="o")
axes[0].set_xlabel("k"); axes[0].set_ylabel("Inertia (within-cluster SSE)"); axes[0].set_title("K-Means elbow curve")
axes[1].plot(list(k_range), sils, marker="s", color="tab:green")
axes[1].set_xlabel("k"); axes[1].set_ylabel("Silhouette score"); axes[1].set_title("Silhouette vs k")
plt.tight_layout(); plt.savefig("../models/clu_elbow.png", bbox_inches="tight"); plt.show()

# Default: the k with the best silhouette. Override K_CHOSEN if the elbow tells a different story.
K_CHOSEN = list(k_range)[int(np.argmax(sils))]
print("K chosen:", K_CHOSEN)


**Observation:** *(where does the elbow bend? does the silhouette peak agree? justify K_CHOSEN in your own words.)*

In [ ]:
kmeans = KMeans(n_clusters=K_CHOSEN, n_init=10, random_state=RANDOM_STATE).fit(X_pca)
km_labels = kmeans.labels_
print("K-Means cluster sizes:", np.bincount(km_labels))


## Agglomerative Hierarchical — dendrogram + linkage comparison

In [ ]:
Z = linkage(X_pca, method="ward")
plt.figure(figsize=(12, 5))
dendrogram(Z, truncate_mode="lastp", p=30, leaf_rotation=90, show_contracted=True)
plt.axhline(y=Z[-(K_CHOSEN - 1), 2], color="red", linestyle="--", label=f"cut for k={K_CHOSEN}")
plt.xlabel("Cluster (sample count in brackets)"); plt.ylabel("Ward distance")
plt.title("Hierarchical clustering dendrogram (Ward linkage, truncated)")
plt.legend(); plt.tight_layout(); plt.savefig("../models/clu_dendrogram.png", bbox_inches="tight"); plt.show()


In [ ]:
# Compare linkage strategies at the same k
link_rows, agg_labels_by_link = [], {}
for link in ["ward", "complete", "average", "single"]:
    lab = AgglomerativeClustering(n_clusters=K_CHOSEN, linkage=link).fit_predict(X_pca)
    agg_labels_by_link[link] = lab
    link_rows.append({"Linkage": link,
                      "Silhouette": silhouette_score(X_pca, lab),
                      "Davies-Bouldin": davies_bouldin_score(X_pca, lab),
                      "Calinski-Harabasz": calinski_harabasz_score(X_pca, lab),
                      "Cluster sizes": np.bincount(lab).tolist()})
link_df = pd.DataFrame(link_rows).set_index("Linkage")
display(link_df)

# Silhouette alone can reward a degenerate result (one giant cluster + a few outliers, typical of
# average/single linkage). So only consider linkages whose smallest cluster holds >= 2% of samples.
min_frac = {l: min(np.bincount(agg_labels_by_link[l])) / len(X_pca) for l in agg_labels_by_link}
valid = [l for l in link_df.index if min_frac[l] >= 0.02]
BEST_LINK = link_df.loc[valid, "Silhouette"].idxmax() if valid else "ward"
print("Linkages with no degenerate (<2%) cluster:", valid)
agg_labels = agg_labels_by_link[BEST_LINK]
print("Linkage used for the final Agglomerative model:", BEST_LINK)


**Observation:** *(which linkage works best and why? single linkage usually chains points into one giant cluster — check the cluster sizes column.)*